In [1]:
import pandas as pd

In [2]:
# concat the two dataframes
df1 = pd.read_csv('../data/foia-504-fy1991-fy2009-asof-251231.csv')
df2 = pd.read_csv('../data/foia-504-fy2010-present-asof-251231.csv')
df_raw = pd.concat([df1, df2], ignore_index=True)
print('# of rows:', len(df_raw))

# of rows: 224331


Drop bad rows of the data.

In [3]:
# remove Canceled, NOT FUNDED, missing loanstatus and remove duplicates
drop_status = ['CANCLD', 'NOT FUNDED']
df_clean = df_raw[~df_raw['loanstatus'].isin(drop_status)].copy()
print('# of rows after removing Canceled/NOT FUNDED:', len(df_clean),
      '| # of removed rows:', len(df_raw) - len(df_clean))

nrow_prev = df_clean.shape[0]
df_clean = df_clean[df_clean['loanstatus'].notna()].copy()
print('# of rows after removing NaN:', len(df_clean),
      '| # of removed rows:', nrow_prev - len(df_clean))

nrow_prev = df_clean.shape[0]
df_clean = df_clean.drop_duplicates(keep='last')
print('# of rows after removing duplicates:', len(df_clean),
      '| # of removed rows:', nrow_prev - len(df_clean))

# check nonnegativity of thirdpartydollars and grosschargeoffamount
nrow_prev = df_clean.shape[0]
df_clean = df_clean[(df_clean['thirdpartydollars'].isna()) | (df_clean['thirdpartydollars'] >= 0)]
df_clean = df_clean[df_clean['grosschargeoffamount'] >= 0]
print('# of rows after removing negative values:', len(df_clean),
      '| # of removed rows:', nrow_prev - len(df_clean))

# remove the grossapproval below 1e3 (too small) and NA
nrow_prev = df_clean.shape[0]
df_clean = df_clean[df_clean['grossapproval'].notna() & (df_clean['grossapproval'] > 1e3)]
print('# of rows after removing grossapproval below 1e3:', len(df_clean),
      '| # of removed rows:', nrow_prev - len(df_clean))

# remove terminmonths below 120 (too short) and NA
nrow_prev = df_clean.shape[0]
df_clean = df_clean[df_clean['terminmonths'].notna() & (df_clean['terminmonths'] >= 120)]
print('# of rows after removing terminmonths below 120:', len(df_clean),
      '| # of removed rows:', nrow_prev - len(df_clean))

# of rows after removing Canceled/NOT FUNDED: 184396 | # of removed rows: 39935
# of rows after removing NaN: 184071 | # of removed rows: 325
# of rows after removing duplicates: 184068 | # of removed rows: 3
# of rows after removing negative values: 184068 | # of removed rows: 0
# of rows after removing grossapproval below 1e3: 184059 | # of removed rows: 9
# of rows after removing terminmonths below 120: 183974 | # of removed rows: 85


In [4]:
# clip the terminmonths at 300 (25 years, the maximum loan term)
df_clean['terminmonths'] = df_clean['terminmonths'].clip(upper=300)
df_clean = df_clean[df_clean['terminmonths'] > 0]

# clip the jobssupported by the 95% upper bound 50
df_clean['jobssupported'] = df_clean['jobssupported'].clip(upper=50)

In [5]:
# convert date columns to datetime
date_cols = [col for col in df_clean.columns if 'date' in col]
df_clean[date_cols] = df_clean[date_cols].apply(pd.to_datetime, errors='coerce')
nrow_prev = df_clean.shape[0]
# firstdisbursementdate no earlier than approvaldate
df_clean = df_clean[(df_clean['firstdisbursementdate'].notna()) & (df_clean['firstdisbursementdate'] > df_clean['approvaldate'])]
# chargeoffdate no earlier than firstdisbursementdate
df_clean = df_clean[(df_clean['chargeoffdate'].isna()) | (df_clean['chargeoffdate'] > df_clean['firstdisbursementdate'])]
# paidinfulldate no earlier than firstdisbursementdate
df_clean = df_clean[(df_clean['paidinfulldate'].isna()) | (df_clean['paidinfulldate'] > df_clean['firstdisbursementdate'])]
# PIF should have paidinfulldate and no chargeoffdate
df_clean = df_clean[(df_clean['loanstatus'] != 'PIF') | ((df_clean['paidinfulldate'].notna()) & (df_clean['chargeoffdate'].isna()))]
# CHGOFF should have chargeoffdate and no paidinfulldate
df_clean = df_clean[(df_clean['loanstatus'] != 'CHGOFF') | ((df_clean['chargeoffdate'].notna()) & (df_clean['paidinfulldate'].isna()))]
# EXEMPT should have no paidinfulldate and no chargeoffdate
df_clean = df_clean[(df_clean['loanstatus'] != 'EXEMPT') | (df_clean['paidinfulldate'].isna()) | (df_clean['chargeoffdate'].isna())]
print('# of rows after removing invalid dates:', len(df_clean),
      '| # of removed rows:', nrow_prev - len(df_clean))

# of rows after removing invalid dates: 183914 | # of removed rows: 60


Deal with missing values.

In [6]:
# remove deliverymethod: 100% missing
df_clean = df_clean.drop(columns=['deliverymethod'])

# encode iffranchise and drop original columns (not much information)
df_clean['iffranchise'] = ((df_clean['franchisename'].notna()) | (df_clean['franchisecode'].notna())).astype(int)
df_clean = df_clean.drop(columns=['franchisename', 'franchisecode'])

# recategorize businessage
df_clean['businessage'] = df_clean['businessage'].replace({
    'Startup, Loan Funds will Open Business': 'Startup',
    'New, Less than 1 Year old': 'Young',
    'New Business or 2 years or less': 'Young',
    'Less than 3 years old but at least 2': 'Young',
    'Less than 4 years old but at least 3': 'Established',
    'Less than 5 years old but at least 4': 'Established',
    'Existing or more than 2 years old': 'Established',
    'Existing, 5 or more years': 'Established',
    'Change of Ownership': 'Ownership'
})
df_clean['businessage'] = df_clean['businessage'].fillna('Unknown')

# keep top 30 of thirdpartylender_name, thirdpartylender_city and encode others as Other
for col in ['thirdpartylender_name', 'thirdpartylender_city']:
    top = df_clean[col].value_counts().nlargest(30).index
    df_clean[col] = df_clean[col].where(df_clean[col].isin(top), 'Other')
    
# # keep thirdpartylender_state with observation > 100 and encode others as Other
# counts = df_clean['thirdpartylender_state'].value_counts()
# df_clean['thirdpartylender_state'] = df_clean['thirdpartylender_state'].replace(counts[counts <= 100].index, 'Other')

# fillna thirdpartylender_name, thirdpartylender_city, thirdpartylender_state
for col in ['thirdpartylender_name', 'thirdpartylender_city', 'thirdpartylender_state']:
    df_clean.loc[(df_clean[col].isna()) & (df_clean['thirdpartydollars'].notna()), col] = df_clean[col].mode().iloc[0]
    df_clean.loc[df_clean[col].isna(), col] = 'NoThirdParty'

# encode thirdpartydollars and fillna with 0
df_clean['ifthirdparty'] = df_clean['thirdpartydollars'].notna().astype(int)
df_clean['thirdpartydollars'] = df_clean['thirdpartydollars'].fillna(0)

# fillna collateralind
df_clean['collateralind'] = df_clean['collateralind'].fillna('Unknown')

# encode naicscode and remove naicsdescription
df_clean['naicscode'] = df_clean['naicscode'].astype('Int64').astype(str)
df_clean['naicscode'] = df_clean['naicscode'].str[:2].replace('<N', 'Unknown')
df_clean = df_clean.drop(columns=['naicsdescription'])
counts = df_clean['naicscode'].value_counts()
df_clean['naicscode'] = df_clean['naicscode'].replace(counts[counts <= 100].index, 'Other')

# remove projectcounty and congressionaldistrict: No much information
df_clean = df_clean.drop(columns=['projectcounty', 'congressionaldistrict'])

# fillna businesstype
df_clean['businesstype'] = df_clean['businesstype'].fillna('Unknown')

# remove cdc_name, cdc_street, cdc_city, cdc_zip, l2locid: duplicated information with cdc_state
df_clean = df_clean.drop(columns=['cdc_name', 'cdc_street', 'cdc_city', 'cdc_zip'])

# encode cdc_state
df_clean['cdc_state'] = df_clean['cdc_state'].fillna('Unknown')
# counts = df_clean['cdc_state'].value_counts()
# df_clean['cdc_state'] = df_clean['cdc_state'].replace(counts[counts <= 100].index, 'Other')

# remove borrstreet, sbadistrictoffice: No much information
df_clean = df_clean.drop(columns=['borrstreet', 'sbadistrictoffice'])

Deal with other columns and remove redundant columns.

In [7]:
# keep top 30 of borrcity and encode others as Other
top = df_clean['borrcity'].value_counts().nlargest(30).index
df_clean['borrcity'] = df_clean['borrcity'].where(df_clean['borrcity'].isin(top), 'Other')

# # keep borrstate with observation > 100 and encode others as Other
# counts = df_clean['borrstate'].value_counts()
# df_clean['borrstate'] = df_clean['borrstate'].replace(counts[counts <= 100].index, 'Other')

# # keep projectstate with observation > 100 and encode others as Other
# counts = df_clean['projectstate'].value_counts()
# df_clean['projectstate'] = df_clean['projectstate'].replace(counts[counts <= 100].index, 'Other')

# remap processingmethod
method_map = {
	'504 Basic':'504',
	'504 Commercial Real Estate Refinance Program':'5RF',
	'504 Refinancing Program':'5RE',
	'ALP Express':'5EX',
	'ALP Express Debt Refi':'5XR',
	'Accredited Lenders Program':'ALP',
	'PCLP Debt Refinance':'5RX',
	'Premier Certified Lenders Program':'PCP'
}
df_clean['processingmethod'] = df_clean['processingmethod'].map(method_map)

# remap subprogram
df_clean['subprogram'] = df_clean['subprogram'].replace({
    'Sec. 504 - Loan Guarantees - Private Sector Financed':'504_regular',
    'Sec. 504 - Premier Certified Lender Program':'504_PCP',
    '504 Refinance':'504_refinance',
    df_clean.iloc[124082]['subprogram']:'Unknown'
})

# remove program, borrname, borrzip, approvalfiscalyear: No much information
df_clean = df_clean.drop(columns=['program', 'borrzip', 'approvalfiscalyear'])

In [8]:
# check the unique values of all categorical variables
for col in df_clean.select_dtypes(include=['string','object']).columns:
    print(col, df_clean[col].unique().shape)

borrname (172182,)
borrcity (31,)
borrstate (54,)
cdc_state (52,)
thirdpartylender_name (31,)
thirdpartylender_city (31,)
thirdpartylender_state (61,)
processingmethod (8,)
subprogram (4,)
naicscode (24,)
projectstate (54,)
businesstype (4,)
businessage (6,)
loanstatus (3,)
collateralind (3,)


In [9]:
df_clean.to_parquet('../data/data_cleaned.parquet', index=False)